# NLP Lab 05 - Solved Tasks

In [1]:
from collections import Counter
from itertools import permutations
import math

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]
test = ["the cat sat on the rug", "the fish sat on the mat"]

# Task 1: Tokenization

In [2]:
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split(" ") + ["</s>"]

def build_vocab(sentences):
    return sorted({word for sentence in sentences for word in tokenize(sentence) if word != "<s>"})

train_tokens = [tokenize(sentence) for sentence in train]
test_tokens = [tokenize(sentence) for sentence in test]
vocab = build_vocab(train)
V = len(vocab)

for i, tokens in enumerate(train_tokens, 1):
    print(f"Train {i}: {tokens}")
for i, tokens in enumerate(test_tokens, 1):
    print(f"Test {i}:  {tokens}")

print("Tokens with markers:", sum(len(tokens) for tokens in train_tokens))
print("Tokens without markers:", sum(len(tokens) - 2 for tokens in train_tokens))
print("V:", V)
print("Sorted vocabulary:", vocab)

Train 1: ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
Train 2: ['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
Train 3: ['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
Train 4: ['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']
Test 1:  ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
Test 2:  ['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']
Tokens with markers: 30
Tokens without markers: 22
V: 11
Sorted vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


# Task 2: Generating N-grams

In [3]:
def get_ngrams(tokens, n):
    if n < 1 or n > len(tokens):
        return []
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

for n in (1, 2, 3):
    print(f"{n}-grams:", get_ngrams(train_tokens[0], n))

for group_name, group in (("Train", train_tokens), ("Test", test_tokens)):
    for i, tokens in enumerate(group, 1):
        m = len(tokens)
        for n in (1, 2, 3):
            assert len(get_ngrams(tokens, n)) == m - n + 1
        print(f"{group_name} {i}: all three N-gram counts verified (m={m})")

print("N larger than sentence length:", get_ngrams(train_tokens[0], len(train_tokens[0]) + 1))

1-grams: [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
2-grams: [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
3-grams: [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]
Train 1: all three N-gram counts verified (m=8)
Train 2: all three N-gram counts verified (m=7)
Train 3: all three N-gram counts verified (m=8)
Train 4: all three N-gram counts verified (m=7)
Test 1: all three N-gram counts verified (m=8)
Test 2: all three N-gram counts verified (m=8)
N larger than sentence length: []


# Task 3: Counting N-grams

In [4]:
unigram_counts = Counter(word for tokens in train_tokens for word in tokens[1:])
bigram_counts = Counter(pair for tokens in train_tokens for pair in get_ngrams(tokens, 2))
trigram_counts = Counter(triple for tokens in train_tokens for triple in get_ngrams(tokens, 3))
prev_counts = Counter()
for (previous, current), count in bigram_counts.items():
    prev_counts[previous] += count

ngram_counts = {1: unigram_counts, 2: bigram_counts, 3: trigram_counts}

def count_of(ngram):
    return ngram_counts.get(len(ngram), Counter())[ngram]

for n, counts in ngram_counts.items():
    print(f"\n{n}-gram counts:")
    for item, count in sorted(counts.items(), key=lambda row: (-row[1], row[0])):
        print(f"{str(item):31s} {count}")

print("\nFive most common bigrams:", bigram_counts.most_common(5))
for n, counts in ngram_counts.items():
    print(f"Distinct {n}-grams: {len(counts)}")
print("Seen bigram ('the', 'cat'):", count_of(("the", "cat")))
print("Unseen bigram ('fish', 'sat'):", count_of(("fish", "sat")))


1-gram counts:
the                             8
</s>                            4
ate                             2
cat                             2
dog                             2
on                              2
sat                             2
bone                            1
fish                            1
mat                             1
rug                             1

2-gram counts:
('<s>', 'the')                  4
('ate', 'the')                  2
('on', 'the')                   2
('sat', 'on')                   2
('the', 'cat')                  2
('the', 'dog')                  2
('bone', '</s>')                1
('cat', 'ate')                  1
('cat', 'sat')                  1
('dog', 'ate')                  1
('dog', 'sat')                  1
('fish', '</s>')                1
('mat', '</s>')                 1
('rug', '</s>')                 1
('the', 'bone')                 1
('the', 'fish')                 1
('the', 'mat')                  1
('the', 'rug')  

# Task 4: Unigram Model

In [5]:
N = sum(unigram_counts.values())

def unigram_prob(w):
    return unigram_counts[w] / N

def unigram_sentence_prob(sentence):
    probability = 1.0
    for w in tokenize(sentence)[1:]:
        probability *= unigram_prob(w)
    return probability

print(f"{'Word':<12} {'Count':>6} {'Probability':>14}")
for w in vocab:
    print(f"{w:<12} {unigram_counts[w]:>6} {unigram_prob(w):>14.4f}")
assert math.isclose(sum(unigram_prob(w) for w in vocab), 1.0)
print("Total probability:", f"{sum(unigram_prob(w) for w in vocab):.4f}")

for sentence in test:
    print(f"P({sentence}) = {unigram_sentence_prob(sentence):.4f} ({unigram_sentence_prob(sentence):.4e})")

words = tuple(test[0].split())
new_order = next(p for p in permutations(words) if p != words)
changed_sentence = " ".join(new_order)
print("Original:", test[0], f"P = {unigram_sentence_prob(test[0]):.4f} ({unigram_sentence_prob(test[0]):.4e})")
print("Reordered:", changed_sentence, f"P = {unigram_sentence_prob(changed_sentence):.4f} ({unigram_sentence_prob(changed_sentence):.4e})")
assert math.isclose(unigram_sentence_prob(test[0]), unigram_sentence_prob(changed_sentence))

Word          Count    Probability
</s>              4         0.1538
ate               2         0.0769
bone              1         0.0385
cat               2         0.0769
dog               2         0.0769
fish              1         0.0385
mat               1         0.0385
on                2         0.0769
rug               1         0.0385
sat               2         0.0769
the               8         0.3077
Total probability: 1.0000
P(the cat sat on the rug) = 0.0000 (2.5499e-07)
P(the fish sat on the mat) = 0.0000 (1.2749e-07)
Original: the cat sat on the rug P = 0.0000 (2.5499e-07)
Reordered: the cat sat on rug the P = 0.0000 (2.5499e-07)


# Task 5: Bigram Model (MLE)

In [6]:
def bigram_prob(prev, w):
    return bigram_counts[(prev, w)] / prev_counts[prev] if prev_counts[prev] else 0.0

def next_word_distribution(prev):
    return {w: bigram_prob(prev, w) for w in vocab if bigram_counts[(prev, w)] > 0}

def bigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    probability = 1.0
    for prev, w in get_ngrams(tokens, 2):
        probability *= bigram_prob(prev, w)
    return probability

def predict_next(prev):
    choices = next_word_distribution(prev)
    return min(choices, key=lambda w: (-choices[w], w)) if choices else None

print("Next-word distribution for 'the':")
for w, p in next_word_distribution("the").items():
    print(f"{w:<12} {p:.4f}")
assert math.isclose(sum(next_word_distribution("the").values()), 1.0)
print("Total:", f"{sum(next_word_distribution('the').values()):.4f}")

for sentence in test:
    print(f"P({sentence}) = {bigram_sentence_prob(sentence):.4f}")

for w in ("<s>", "the", "cat", "dog", "sat", "ate", "fish", "rug"):
    print(f"After {w:<5} -> {predict_next(w)}")

Next-word distribution for 'the':
bone         0.1250
cat          0.2500
dog          0.2500
fish         0.1250
mat          0.1250
rug          0.1250
Total: 1.0000
P(the cat sat on the rug) = 0.0156
P(the fish sat on the mat) = 0.0000
After <s>   -> the
After the   -> cat
After cat   -> ate
After dog   -> ate
After sat   -> on
After ate   -> the
After fish  -> </s>
After rug   -> </s>


# Task 6: Trigram Model

In [7]:
def trigram_prob(w1, w2, w3):
    return trigram_counts[(w1, w2, w3)] / bigram_counts[(w1, w2)] if bigram_counts[(w1, w2)] else 0.0

def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    probability = bigram_prob(tokens[0], tokens[1])
    for w1, w2, w3 in get_ngrams(tokens, 3):
        probability *= trigram_prob(w1, w2, w3)
    return probability

for sentence in test:
    print(f"Trigram P({sentence}) = {trigram_sentence_prob(sentence):.4f}")

print(f"{'Model':<12} {'Distinct':>10}")
for n, label in ((1, "Unigram"), (2, "Bigram"), (3, "Trigram")):
    print(f"{label:<12} {len(ngram_counts[n]):>10}")

for sentence in test:
    tokens = tokenize(sentence)
    unseen_bi = [pair for pair in get_ngrams(tokens, 2) if bigram_counts[pair] == 0]
    unseen_tri = [triple for triple in get_ngrams(tokens, 3) if trigram_counts[triple] == 0]
    print(f"\n{sentence}")
    print("Unseen bigrams:", unseen_bi)
    print("Unseen trigrams:", unseen_tri)
    if len(unseen_tri) > len(unseen_bi):
        print("More unseen: Trigram model")
    elif len(unseen_tri) < len(unseen_bi):
        print("More unseen: Bigram model")
    else:
        print("Equal unseen N-grams")

Trigram P(the cat sat on the rug) = 0.1250
Trigram P(the fish sat on the mat) = 0.0000
Model          Distinct
Unigram              11
Bigram               18
Trigram              19

the cat sat on the rug
Unseen bigrams: []
Unseen trigrams: []
Equal unseen N-grams

the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
More unseen: Trigram model


# Task 7: Zero-probability Problem

In [8]:
def unseen_bigrams(sentence):
    return [pair for pair in get_ngrams(tokenize(sentence), 2) if bigram_counts[pair] == 0]

def safe_log(p):
    return math.log(p) if p > 0 else -math.inf

for sentence in test:
    print(f"{sentence}")
    print("Unseen bigrams:", unseen_bigrams(sentence))
    print("Bigram MLE probability:", f"{bigram_sentence_prob(sentence):.4f}")

assert any(bigram_sentence_prob(sentence) == 0 for sentence in test)
words_only = [w for w in vocab if w != "</s>"]
possible_pairs = len(words_only) ** 2
seen_pairs = sum(bigram_counts[(a, b)] > 0 for a in words_only for b in words_only)
unseen_pairs = possible_pairs - seen_pairs
print(f"Unseen pairs: {unseen_pairs}/{possible_pairs} ({100 * unseen_pairs / possible_pairs:.4f}%)")

log_probability = sum(safe_log(bigram_prob(prev, w)) for prev, w in get_ngrams(tokenize(test[1]), 2))
print("Second test sentence log probability:", log_probability)

the cat sat on the rug
Unseen bigrams: []
Bigram MLE probability: 0.0156
the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Bigram MLE probability: 0.0000
Unseen pairs: 87/100 (87.0000%)
Second test sentence log probability: -inf


# Task 8: Smoothing

In [9]:
def bigram_prob_addone(prev, w):
    return (bigram_counts[(prev, w)] + 1) / (prev_counts[prev] + V)

def bigram_prob_addk(prev, w, k):
    return (bigram_counts[(prev, w)] + k) / (prev_counts[prev] + k * V)

print(f"{'Probability':<18} {'MLE':>10} {'Add-one':>12}")
for prev, w in (("fish", "sat"), ("the", "cat")):
    print(f"P({w}|{prev}):{'':<7} {bigram_prob(prev, w):>10.4f} {bigram_prob_addone(prev, w):>12.4f}")

for prev in ["<s>"] + vocab:
    assert math.isclose(sum(bigram_prob_addone(prev, w) for w in vocab), 1.0)
print("All add-one next-word distributions sum to 1.0000")

print(f"{'k':<8} {'P(sat|fish)':>15}")
for k in (1, 0.5, 0.1, 0.01):
    print(f"{k:<8} {bigram_prob_addk('fish', 'sat', k):>15.4f}")

Probability               MLE      Add-one
P(sat|fish):            0.0000       0.0833
P(cat|the):            0.2500       0.1579
All add-one next-word distributions sum to 1.0000
k            P(sat|fish)
1                 0.0833
0.5               0.0769
0.1               0.0476
0.01              0.0090


# Task 9: Perplexity and Model Comparison

In [10]:
def perplexity(sentences, prob_fn):
    total_log_probability = 0.0
    token_count = 0
    for sentence in sentences:
        for prev, w in get_ngrams(tokenize(sentence), 2):
            p = prob_fn(prev, w)
            if p == 0:
                return math.inf
            total_log_probability += math.log(p)
            token_count += 1
    return math.exp(-total_log_probability / token_count) if token_count else math.inf

def unigram_prob_addone(w):
    return (unigram_counts[w] + 1) / (N + V)

models = {
    "Bigram MLE": perplexity(test, bigram_prob),
    "Bigram add-one": perplexity(test, bigram_prob_addone),
    "Unigram add-one": perplexity(test, lambda prev, w: unigram_prob_addone(w))
}
print(f"{'Model':<20} {'Perplexity':>12}")
for label, value in models.items():
    print(f"{label:<20} {value:>12.4f}")

k_values = (1, 0.5, 0.1, 0.01)
test_scores = {k: perplexity(test, lambda prev, w, k=k: bigram_prob_addk(prev, w, k)) for k in k_values}
print(f"\n{'k':<8} {'Test perplexity':>18}")
for k, score in test_scores.items():
    print(f"{k:<8} {score:>18.4f}")
best_test_k = min(test_scores, key=test_scores.get)
print("Best k from test set:", best_test_k)

validation_train_tokens = [tokenize(sentence) for sentence in train[:3]]
validation_bigram_counts = Counter(pair for tokens in validation_train_tokens for pair in get_ngrams(tokens, 2))
validation_prev_counts = Counter()
for (prev, w), count in validation_bigram_counts.items():
    validation_prev_counts[prev] += count

def validation_addk_prob(prev, w, k):
    return (validation_bigram_counts[(prev, w)] + k) / (validation_prev_counts[prev] + k * V)

validation_scores = {
    k: perplexity([train[3]], lambda prev, w, k=k: validation_addk_prob(prev, w, k))
    for k in k_values
}
print(f"\n{'k':<8} {'Validation perplexity':>23}")
for k, score in validation_scores.items():
    print(f"{k:<8} {score:>23.4f}")
best_validation_k = min(validation_scores, key=validation_scores.get)
print("Best k from validation:", best_validation_k)
print("Test perplexity using validation-selected k (trained on first 3):",
      f"{perplexity(test, lambda prev, w: validation_addk_prob(prev, w, best_validation_k)):.4f}")
print("Test perplexity using test-selected k (trained on first 3):",
      f"{perplexity(test, lambda prev, w: validation_addk_prob(prev, w, best_test_k)):.4f}")
print("Test perplexity using test-selected k (trained on all 4):",
      f"{test_scores[best_test_k]:.4f}")
print("Fixed vocabulary size used in both models:", V)

Model                  Perplexity
Bigram MLE                    inf
Bigram add-one             5.7969
Unigram add-one            9.1370

k           Test perplexity
1                    5.7969
0.5                  4.5703
0.1                  2.9559
0.01                 2.6272
Best k from test set: 0.01

k          Validation perplexity
1                         8.5857
0.5                       8.0163
0.1                       8.0260
0.01                     13.1304
Best k from validation: 0.5
Test perplexity using validation-selected k (trained on first 3): 4.4539
Test perplexity using test-selected k (trained on first 3): 2.4258
Test perplexity using test-selected k (trained on all 4): 2.6272
Fixed vocabulary size used in both models: 11
